### PlayWright
- 자바스크립트에서 주로 데이터 수집할 때 사용하는 것을
- 파이썬에서 사용 가능하도록 한 패키지
- 사실 주피터 노트북에서는 작동 잘 안해서, 임의로 사용하기 쉽게 수정함 -> =아래 코드 활용

### 1. 헬퍼 코드 준비

In [2]:
import asyncio
import sys
import threading
from playwright.async_api import async_playwright

_bg_loop = None
_bg_thread = None


def _ensure_bg_loop():
    """ProactorEventLoop를 백그라운드 스레드에서 실행"""
    global _bg_loop, _bg_thread
    if _bg_loop is None or not _bg_loop.is_running():
        if sys.platform == "win32":
            _bg_loop = asyncio.ProactorEventLoop()
        else:
            _bg_loop = asyncio.new_event_loop()
        _bg_thread = threading.Thread(target=_bg_loop.run_forever, daemon=True)
        _bg_thread.start()
    return _bg_loop


async def run_pw(coro):
    """Playwright 비동기 명령을 백그라운드 루프에서 실행
    사용법: result = await run_pw(page.goto('https://naver.com'))
    """
    loop = _ensure_bg_loop()
    future = asyncio.run_coroutine_threadsafe(coro, loop)
    while not future.done():
        await asyncio.sleep(0.05)
    return future.result()


print("Playwright 헬퍼 준비 완료")

Playwright 헬퍼 준비 완료


### 2. 브라우저 띄우기
- 브라우저를 띄워서 제어 -> 브라우저는 작업동안에는 끄면 안됨!
- 작업이 다 끝나면 코드로 브라우저 종료시킬 예정.

In [3]:
# 비동기모드로 실행하기
pw = await run_pw(async_playwright().start())

# 크롬 브라우저를 실행
browser = await run_pw(pw.chromium.launch(headless=False))

page = await run_pw(browser.new_page())

### 3. 페이지 접속

In [5]:
await run_pw(page.goto("https://www.naver.com"))

# 대기코드
await run_pw(page.wait_for_timeout(2000)) # 최대 2초까지 대기

In [6]:
title = await run_pw(page.title())
print(title)

NAVER


In [9]:
search_box = page.get_by_placeholder("검색어를 입력해 주세요.")

# 검색어 입력
await run_pw(search_box.fill("조승우"))
# 대기코드
await run_pw(page.wait_for_timeout(2000))

In [10]:
# 엔터 쳐야 검색 가능! (그전까진 입력만 함)
await run_pw(search_box.press("Enter"))
# 대기코드
await run_pw(page.wait_for_timeout(2000)) 

### 4. 뉴스 검색

In [11]:
url = "https://search.naver.com/search.naver?where=news&query=조승우"
await run_pw(page.goto(url))
await run_pw(page.wait_for_timeout(2000)) 

In [ ]:
# 제목들 추출해보기
titles = page.locator('a[data-heatmap-target=".tit"]') # a로 시작하는 이유: f12 창에 타이틀 창이 a로 시작하기 때문...
# 코드 자체에 " 있으니 겉에 씌울 땐 '로

#갯수 확인해보기
count = await run_pw(titles.count())
print(count)
# 이 결과는 스크롤 얼마나 하냐에 따라 다르다!
# 즉, 전부 다 가져오려면 스크롤도 자동화해야한다!

19


In [ ]:
# 내용들 추출해보기

contents = page.locator('a[data-heatmap-target=".body"]') # a로 시작하는 이유: f12 창에 타이틀 창이 a로 시작하기 때문...

content_count = await run_pw(contents.count())
print(content_count)

# 결과 차이나는 이유: 페이지 자체 특성상, 제목만 있는 경우들 있음

10


1. 제목을 찾는다
2. 상위 태그를 찾는다 (f12 창에서)
3. 상위 태그에서 제목과 내용 쌍이 있는지 확인한다
4. 내용이 없을 경우에는 문자열(빈값: "")로 처리한다

In [23]:
# 한 건 테스트
# 1. 제목을 찾는다.

title = titles.nth(0) # 첫 번째꺼 찾는 방법
await run_pw(title.text_content())

# 2. 상위 태그 찾는다
parent = title.locator("xpath=..") # 상위 찾는 방법

# 3. 제목과 내용 쌍을 확인해보기 - 내용이 있는지만 확인해보면 됨
body = parent.locator('a[data-heatmap-target=".body"]')

# 개수 세어보면 됨 -> 있으면 1, 없으면 0

has_body = await run_pw(body.count())
print(has_body)

1


In [ ]:
# 아래처럼 저장해야 한다.
total_data = [
    {
        "title" : "제목1",
        "content" : "내용1"
    },
    {
        "title" : "제목2",
        "content" : "내용2"
    }
]

In [ ]:
# 총 title과 content를 for문을 이용해서 반복
total_data = []

for i in range(count):

    title = titles.nth(i) # i 번째 요소 찾아야 하므로

    # 1. 제목을 찾는다
    title_content = await run_pw(title.text_content())

    # 2. 상위 태그 찾는다
    parent = title.locator("xpath=..") # 상위 찾는 방법

    # 3. 제목과 내용 쌍을 확인해보기 - 내용이 있는지만 확인해보면 됨
    body = parent.locator('a[data-heatmap-target=".body"]')

    # 개수 세어보면 됨 -> 있으면 1, 없으면 0

    has_body = await run_pw(body.count())

    # 내용이 있을 때에는 그 내용 가져오고, 없을 때에는 "" 이렇게 표시하기

    if has_body > 0:    # 내용 있을 때
        news_content = await run_pw(body.text_content())
    else:
        news_content = ""

    total_data.append(
        {
            "title" : title_content,
            "content" : news_content
        }
    )
    print(i, "번째 작업 중입니다.")

0 번째 작업 중입니다.
1 번째 작업 중입니다.
2 번째 작업 중입니다.
3 번째 작업 중입니다.
4 번째 작업 중입니다.
5 번째 작업 중입니다.
6 번째 작업 중입니다.
7 번째 작업 중입니다.
8 번째 작업 중입니다.
9 번째 작업 중입니다.
10 번째 작업 중입니다.
11 번째 작업 중입니다.
12 번째 작업 중입니다.
13 번째 작업 중입니다.
14 번째 작업 중입니다.
15 번째 작업 중입니다.
16 번째 작업 중입니다.
17 번째 작업 중입니다.
18 번째 작업 중입니다.


In [26]:
import pandas as pd

df = pd.DataFrame(total_data)
df

,title,content
0,"‘슈퍼 집돌이’ 조승우, 알고 보니 반전 성격 “촬영장 가면 I→E로 바...새 창 열림",배우 조승우가 ‘슈퍼 집돌이’다운 일상과 함께 일할 때만큼은 180도 달라지는 반전...
1,"조승우 ""애교·웃음 많은 편...집에선 존재감 없지만 현장 가면 'E'로 변...새...",
2,"조승우→변요한, 20년 '타짜' 마지막 승부…추석 극장가 다시 접수새 창 열림",2006년 조승우의 '타짜'를 시작으로 추석마다 관객을 찾아온 시리즈가 올해 마지막...
3,"조승우, '타짜4' 변요한에 보낸 문자…""정 마담 김혜수 대사로 답장""새 창 열림",출연을 확정한 이후에는 '타짜' 1편에서 조승우가 맡았던 캐릭터와 관련해 직접 문자...
4,"변요한, '타짜4' 출연 확정 후 조승우에 문자 ""정마담 대사로 답장""[화보...새...",
5,"조승우, '타짜4' 변요한에 보낸 문자 공개 ""정 마담 김혜수 대사가..""새 창 열림",
6,"변요한, ‘타짜: 벨제붑의 노래’ 출연 앞두고 조승우에 연락…돌아온 ...새 창 열림",
7,"'타짜4' 변요한·노재원 화보 공개 ""조승우 응원에 용기 얻었다""새 창 열림",
8,"조승우 SP AERO 대표, 경상국립대에 발전기금 기탁새 창 열림",경상국립대는 ㈜SP AERO 조승우 대표가 최근 학생 장학금 지원을 위해 발전기금 ...
9,조승우 “청년도 중년도 아닌 위치…조금 더 숙성될 필요 있어” [화보...새 창 열림,배우 조승우의 화보가 공개됐다. 조승우가 매거진 ‘아레나 옴므 플러스’ 9월호의 표...


#### 실습 - 유튜브 해보기(개인과제)

In [27]:
await run_pw(page.goto("https://www.youtube.com/"))

# 대기코드
await run_pw(page.wait_for_timeout(2000)) # 최대 2초까지 대기

In [ ]:
search_box = page.get_by_placeholder("검색")

# 검색어 입력
await run_pw(search_box.fill("hip hop"))
# 대기코드
await run_pw(page.wait_for_timeout(2000))

# 엔터 쳐야 검색 가능! (그전까진 입력만 함)
await run_pw(search_box.press("Enter"))
# 대기코드
await run_pw(page.wait_for_timeout(2000)) 

In [ ]:
# 가장 상단 들어가보기?
video_titles = page.locator('a[id="video-title"]') # a로 시작하는 이유: f12 창에 타이틀 창이 a로 시작하기 때문...
first_video = video_titles.nth(0)
print(await run_pw(first_video.text_content()))

await run_pw(first_video.click())
print(page.url)


            
            Playlist 이걸 안듣는 너만 손해! 도입부 개쩌는 외힙 🔥 | Best Hip-Hop Mix [36곡]
          
https://www.youtube.com/watch?v=Y6LO2Tl9axM&list=RDY6LO2Tl9axM&start_radio=1


In [35]:
await run_pw(browser.close())